# Object detection

Object detection is all about drawing boxes (called bounding boxes) around objects of interest in a picture.

Common applications:

1)  Counting — Find out how many instances of an object are in an image.

2)  Tracking — Track how objects move in a scene over time by performing object detection on every frame of a movie.

3)  Cropping — Identify the area of an image that contains an object of interest to crop it and send a higher-resolution version of the image patch to a classifier or an Optical Character Recognition (OCR) model.

<img src="./images/dl_cnn_19.png" width=1000 />

### Difference between Image segmentation and Object detection

Segmentation is a strict superset of detection — it gives full pixel-level masks, from which bounding boxes can be derived, but this comes with higher compute cost.

Segmentation requires expensive labeling — pixel-perfect masks are far harder to annotate than simple bounding boxes.

Use detection when pixel-level detail isn’t needed — it’s faster, cheaper, and ideal for tasks like object counting or locating objects without fine boundaries.

### Object detection architectures

There are two broad categories of object detection architectures:

Two-stage detectors, which first extract region proposals, known as Region-based Convolutional Neural Networks (R-CNN) models

Single-stage detectors, such as RetinaNet or the You Only Look Once family of models

#### 1) R-CNN (Region based CNN)

Input Image: Start with a single input image containing one or more objects.

Region Proposal Generation: Use Selective Search(greedy search) to generate around 2,000 region proposals (potential object locations).

Warp & Feature Extraction: Each proposed region is cropped and resized (warped) to a fixed size. Then pass each region through a CNN to extract feature vectors.

Region Classification: Use the extracted features to classify each region using SVMs into object categories (e.g. person, car) or background.

<img src="./images/dl_cnn_20.png" width=1000 />

<img src="./images/dl_cnn_21.png" width=1000 />

The input size for AlexNet is (227, 227, 3), meaning each input image must be resized to these dimensions. Consequently, whether the region proposals are small or large, they need to be adjusted accordingly to fit the specified input size.

From the above architecture, we remove the final softmax layer to obtain a (1, 4096) feature vector.

This feature vector is then fed into both the Support Vector Machine (SVM) for classification and the bounding box regressor for improved localization.

The SVM model takes the feature vector produced by the previous CNN architecture and outputs a confidence score indicating the likelihood of an object being present in that region.

To accurately locate the bounding box within the image, we utilize a scale-invariant linear regression model known as the bounding box regressor.

For training this model, we use pairs of predicted and ground truth values for four dimensions of localization: (x,y,w,h). Here, 
x and y represent the pixel coordinates of the center of the bounding box, while w and h indicate the width and height of the bounding boxes, respectively.

<img src="./images/dl_cnn_22.png" width=1000 />

To further optimize detection, R-CNNs apply Non-Maximum Suppression (NMS):

1)  Remove proposals with confidence scores below a threshold (e.g., 0.5).

2)  Select the highest-probability region among candidates for each object.

3)  Discard overlapping regions with an IoU (Intersection over Union) above 0.5 to eliminate duplicate detections. IoU =  Area of Union / Area of Overlap
​


<img src="./images/dl_cnn_23.png" width=1000 />

However, this architecture is very slow to train and takes ~ 49 sec to generate test results on a single image 

#### 2)  Fast R-CNN

Single Stage Processing: Instead of extracting features for each region proposal independently, Fast R-CNN processes the entire image once through the CNN to generate a feature map. The region proposals are then extracted from this shared feature map.


Softmax Classifier: Fast R-CNN replaces the SVM with a softmax classifier, allowing for end-to-end training of the network.


Improved Bounding Box Regression: Fast R-CNN enhances the bounding box regression process, leading to better localization accuracy.

#### 3)  Faster R-CNN

Faster R-CNN further advances the R-CNN framework by incorporating a Region Proposal Network (RPN).

Region Proposal Network: The RPN generates high-quality region proposals directly from the feature maps produced by the CNN, eliminating the need for selective search.

Shared Convolutional Features: Both the RPN and the detection network share the convolutional features, significantly reducing computation time.

Improved Speed: Faster R-CNN achieves real-time processing speeds of around 0.1 seconds per image while maintaining high detection accuracy.

Input Image  
    ↓  
Convolutional Neural Network (CNN Backbone)  
    ↓  
Shared Feature Map  
    ↓  
Region Proposal Network (RPN)  
    ↓  
Region Proposals (ROIs)  
    ↓  
ROI Pooling / ROI Align  
    ↓  
Detection Head (Classifier + Bounding Box Regressor)

#### 4) Mask R-CNN

Building upon Faster R-CNN, Mask R-CNN was introduced to extend the model to perform instance segmentation. Key features include:

Segmentation Masks: In addition to bounding boxes, Mask R-CNN predicts a segmentation mask for each detected object, providing pixel-level accuracy. It's an instance segmentation model.

Feature Pyramid Networks (FPN): Mask R-CNN incorporates FPNs to improve performance on objects at different scales, enhancing detection accuracy for small objects.

Use cases where box detection is NOT enough:
<pre>
	•	Medical imaging (tumor boundaries)
	•	Autonomous driving (pedestrian outlines)
</pre>

#### Challenges of R-CNN based architectures

R-CNN:
<pre>
	•	Rigid Selective Search: Uses a fixed, non-learnable selective search algorithm for region proposals, often producing suboptimal candidate regions.
	•	Extremely Slow Training: ~2,000 proposals per image must be fed through a CNN individually; CNN, SVM classifier, and bounding-box regressor all require separate training steps.
	•	No Real-Time Performance: Processing a single image can take ~50 seconds, making it unusable for time-critical applications.
	•	Heavy Memory Usage: Stores feature maps for thousands of proposals per image, requiring massive disk space and RAM.
</pre>

Fast R-CNN

<pre>
	•	Still Depends on Selective Search: Region proposals are still generated by the slow, non-learnable selective search method, which becomes the bottleneck.
	•	Single-Stage Training but Slow Proposal Generation: Although training is simplified, inference speed is still limited by region proposal generation.
	•	Not Real-Time: Typical processing is still around 1–2 seconds per image — faster but still insufficient for real-time systems.
</pre>

Faster R-CNN

<pre>
	•	High Computation Cost: Although RPN replaces selective search, the backbone CNN + RPN + detection head still demands significant GPU compute.
	•	Not True Real-Time: Speeds of ~5–7 FPS (0.1–0.2 seconds per image) are much better, but still slower than YOLO and other single-shot detectors.
</pre>

Mask R-CNN

Mask R-CNN provides powerful instance segmentation but remains computationally heavy, slow for real-time applications, complex to implement, memory-intensive, and reliant on high-quality mask annotations.



### Single-stage detectors - YOLO

The main families of single-stage detectors are RetinaNet, Single Shot MultiBox Detectors (SSD), and the You Only Look Once family, abbreviated as YOLO.

YOLO is very popular among all - there are 12 versions released so far.

YOLO treats detection as a regression problem, not a two-stage classification problem.

For each detected object:

(x_min, y_min, x_max, y_max)

confidence_score

class_id

YOLO looks at the whole image one single time, like a human glancing at a photo and instantly spotting objects.

YOLO says:

<pre>

“Let me divide the image into small neighborhoods.
Each neighborhood will check:
— Is there an object here?
— What object is it?
— Where exactly is it located?”

</pre>

So every neighborhood becomes a mini-detector that works in parallel.

After all neighborhoods vote, YOLO:

“I’ll clean up duplicate boxes and only keep the best ones.”

This makes YOLO simple, fast, and real-time.

It never needs to “hunt around” for objects — it just predicts directly.

However, single-shot object detection is generally less accurate than other methods, and it’s less effective in detecting small objects. 

Such algorithms can be used to detect objects in real time in resource-constrained environments.

<img src="./images/dl_cnn_24.png" width=1000 />

<img src="./images/dl_cnn_25.png" width=1000 />

In [1]:
from ultralytics import YOLO
import os, shutil, random
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/manojkumar_rajendran/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# Convert trimaps → YOLO bounding box labels

import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

images_dir = Path("dataset/images")
masks_dir = Path("dataset/annotations/trimaps")

all_images = sorted(images_dir.glob("*.jpg"))

print("Total images:", len(all_images))

Total images: 7390


In [3]:
train_imgs, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)

base_dir = Path("dataset/pet_yolo")

for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    (base_dir / sub).mkdir(parents=True, exist_ok=True)

In [4]:
# Rough split: cat breeds vs dog breeds from dataset description
cat_breeds = {
    "Abyssinian", "Bengal", "Birman", "Bombay", "British_Shorthair",
    "Egyptian_Mau", "Maine_Coon", "Persian", "Ragdoll", "Russian_Blue",
    "Siamese", "Sphynx"
}

def get_class_id_from_name(fname: str) -> int:
    """
    fname: 'Abyssinian_12.jpg' or 'chihuahua_99.jpg'
    We treat cat breeds above as class 0, dogs as class 1.
    """
    stem = Path(fname).stem  # 'Abyssinian_12'
    breed = "_".join(stem.split("_")[:-1])  # 'Abyssinian'
    if breed in cat_breeds:
        return 0  # cat
    else:
        return 1  # dog

def mask_to_bbox(mask_arr: np.ndarray):
    """
    mask_arr: HxW array where pet pixels != 0 (we'll treat 1,2,3 as pet/border)
    returns (x_min, y_min, x_max, y_max) in pixel coords, or None if empty
    """
    ys, xs = np.where(mask_arr > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    return x_min, y_min, x_max, y_max

def save_yolo_label(img_path: Path, split: str):
    # Find corresponding mask: same name but .png
    mask_path = masks_dir / (img_path.stem + ".png")
    if not mask_path.exists():
        return

    # Load image & mask
    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    mask = Image.open(mask_path)
    mask_arr = np.array(mask)

    # In trimaps: 1,2,3 correspond to pet/foreground/border; we treat non-zero as pet
    pet_mask = (mask_arr > 0).astype(np.uint8)

    bbox = mask_to_bbox(pet_mask)
    if bbox is None:
        return

    x_min, y_min, x_max, y_max = bbox

    # convert to YOLO normalized format
    x_center = (x_min + x_max) / 2.0 / w
    y_center = (y_min + y_max) / 2.0 / h
    bw = (x_max - x_min) / w
    bh = (y_max - y_min) / h

    class_id = get_class_id_from_name(img_path.name)

    # write label file
    label_path = base_dir / "labels" / split / (img_path.stem + ".txt")
    with open(label_path, "w") as f:
        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}\n")

    # copy image to yolo folder
    dest_img = base_dir / "images" / split / img_path.name
    shutil.copy2(img_path, dest_img)

In [5]:
for p in train_imgs:
    save_yolo_label(p, "train")

for p in val_imgs:
    save_yolo_label(p, "val")

print("Train images:", len(list((base_dir/"images/train").glob("*.jpg"))))
print("Val images:", len(list((base_dir/"images/val").glob("*.jpg"))))

Train images: 5912
Val images: 1478


In [6]:
yaml_text = """
path: pet_yolo
train: images/train
val: images/val

names:
  0: cat
  1: dog
"""

with open(base_dir / "pet.yaml", "w") as f:
    f.write(yaml_text)

In [ ]:
model = YOLO("yolov8n.pt")  # small model; can use yolov8s.pt / m.pt / l.pt

# I tried to RUN in Macbook Pro and I could see my system getting abnormally overheated in just 5 minutes !

model.train(
    data=str(base_dir / "pet.yaml"),
    epochs=10,
    imgsz=640,
    batch=16,
    project="runs_pet_yolo",
    name="yolov8n_oxfordIIIT",
)

Ultralytics 8.3.229 🚀 Python-3.11.14 torch-2.9.1 CPU (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/pet_yolo/pet.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_oxfordIIIT2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

KeyboardInterrupt: 